In [1]:
# ============================================================
# Cell 1: 环境初始化
# ============================================================
import os, sys, json

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.ci.landscape_service import (
    compare_competitor_programs,
    map_engineering_landscape,
    compare_engineering_strategies,
    identify_landscape_white_space,
)

driver = get_driver()
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# ============================================================
# Cell 2: 查出可对比的 DevelopmentProgram 列表
# ============================================================
with driver.session() as s:
    progs = [dict(r) for r in s.run("""
        MATCH (d:DevelopmentProgram)
        OPTIONAL MATCH (d)-[:TARGETS_PATHOGEN]->(p:Pathogen)
        RETURN d.program_id AS pid, d.canonical_name AS name,
               collect(DISTINCT p.species) AS pathogens
        ORDER BY d.canonical_name
        LIMIT 10
    """)]

print(f"共 {len(progs)} 个项目：")
for p in progs:
    print(f"  {p['pid']}  {p['name']}  ->  {p['pathogens']}")

共 10 个项目：
  CI:PROG:A55770C2  Athinia  ->  []
  CI:PROG:CE12D761  BX004  ->  ['Klebsiella pneumoniae']
  CI:PROG:039E476A  BX011  ->  ['Staphylococcus aureus']
  CI:PROG:6774B789  BX211  ->  ['Staphylococcus aureus']
  CI:PROG:7B79E20C  Data42  ->  []
  CI:PROG:CA3E661C  LBP-PA01  ->  ['Pseudomonas aeruginosa']
  CI:PROG:88B824DA  ListShield  ->  []
  CI:PROG:79E345AA  PhageBank-CF  ->  ['Pseudomonas aeruginosa']
  CI:PROG:981FB965  PhageBank-DFO  ->  ['Klebsiella pneumoniae']
  CI:PROG:017BDA3A  PhageBank-PJI  ->  ['Staphylococcus aureus']


In [3]:
# ============================================================
# Cell 3: PRD 13.2 竞争格局矩阵（compare_competitor_programs）
# ============================================================
program_ids = [p["pid"] for p in progs[:5]]
matrix = compare_competitor_programs(driver, program_ids)

print("=== as_of_date ===", matrix["as_of_date"])
print("=== dimensions ===", matrix["dimensions"])

print("\n=== programs ===")
for p in matrix["programs"]:
    print(json.dumps(p, ensure_ascii=False))

print("\n=== matrix ===")
print("x_axis:", matrix["matrix"]["x_axis"])
print("y_axis:", matrix["matrix"]["y_axis"])
for row in matrix["matrix"]["cells"]:
    for cell in row:
        if cell["count"] > 0:
            print(f"  {cell['strategy_type']} × {cell['pathogen_species']} -> {cell['count']} 个项目")

=== as_of_date === 2026-09-30
=== dimensions === ['target_pathogen', 'clinical_indication', 'development_stage', 'modality', 'engineering_strategy', 'trial_status', 'patent_coverage', 'evidence_maturity']

=== programs ===
{"program_id": "CI:PROG:A55770C2", "canonical_name": "Athinia", "organization": "Merck KGaA", "target_pathogen": [], "clinical_indication": ["Diabetic foot osteomyelitis", "Cystic fibrosis lung infection"], "development_stage": "commercial", "modality": "software", "engineering_strategy": ["payload_delivery", "biofilm_disruption", "lysis_enhancement", "tail_fiber_engineering", "host_range_expansion"], "trial_status": {"total": 2, "by_status": {"active": 1, "completed": 1}, "by_phase": {"phase_2": 1, "phase_1_2": 1}, "active": [{"trial_id": "CI:TRIAL:1A53E4F0", "nct_id": "NCT06098765", "phase": "phase_2", "status": "active"}]}, "patent_coverage": {"total_families": 3, "by_legal_status": {"pending": 2, "granted": 1}}, "evidence_maturity": "in_vitro"}
{"program_id": "CI

In [4]:
# ============================================================
# Cell 4: PRD 13.4 工程化技术图谱（map_engineering_landscape）
# ============================================================
with driver.session() as s:
    pid_row = s.run("""
        MATCH (ec:EngineeredPhageConstruct)-[:TARGETS]->(p:Pathogen)
        RETURN p.pathogen_id AS pid, p.species AS species, count(ec) AS c
        ORDER BY c DESC LIMIT 1
    """).single()
    if pid_row:
        target_pid = pid_row["pid"]
        target_sp = pid_row["species"]
        print(f"使用 pathogen: {target_sp} ({target_pid})")
    else:
        target_pid = None
        print("⚠️ 无构建体关联病原")

landscape = map_engineering_landscape(driver, target_pathogen_id=target_pid)

print("\n=== known_constructs ===")
for c in landscape["known_constructs"]:
    print(f"  {c['public_name']} [{c['construct_status']}] "
          f"策略={c['strategies']} 靶向={c['pathogens']}")

print("\n=== organizations ===", landscape["organizations"])
print("=== evidence_maturity_distribution ===", landscape["evidence_maturity_distribution"])

print("\n=== white_spaces ===")
for ws in landscape["white_spaces"][:10]:
    print(f"  {ws['strategy_type']} × {ws['pathogen_species']}  [{ws['coverage']}]")

print("\n=== uncertainties ===")
for u in landscape["uncertainties"]:
    print(f"  - {u}")

print("\n=== disclaimer ===")
print(landscape["disclaimer"])

使用 pathogen: Pseudomonas aeruginosa (PATH-004)

=== known_constructs ===
  vB_Pae_PLD_010 [proposed] 策略=['payload_delivery'] 靶向=['Pseudomonas aeruginosa']
  vB_Pae_BFD_007 [in_vitro_tested] 策略=['biofilm_disruption'] 靶向=['Pseudomonas aeruginosa']
  vB_Pae_LYE_005 [in_vitro_tested] 策略=['lysis_enhancement'] 靶向=['Pseudomonas aeruginosa']
  vB_Pae_HRE_002 [in_vitro_tested] 策略=['host_range_expansion'] 靶向=['Pseudomonas aeruginosa']

=== organizations === ['Merck KGaA']
=== evidence_maturity_distribution === {'conceptual': 0, 'in_vitro': 1, 'in_vivo': 0, 'clinical': 0}

=== white_spaces ===
  tail_fiber_engineering × Pseudomonas aeruginosa  [empty]

=== uncertainties ===
  - 当前证据成熟度以体外为主，临床转化存在不确定性
  - 部分构建体仅处于 proposed 状态，未见实测数据

=== disclaimer ===
White space 表示在当前已收集、截至指定日期的公开证据中覆盖较少的区域。它不等于：市场上绝对无人布局、不存在未公开项目、该区域必然有商业价值、该区域不存在专利风险。


In [5]:
# ============================================================
# Cell 5: PRD 13.5 技术路线对比（compare_engineering_strategies）
# ============================================================
with driver.session() as s:
    strategies = [dict(r) for r in s.run("""
        MATCH (es:EngineeringStrategy)
        RETURN es.strategy_id AS sid, es.strategy_type AS stype
        ORDER BY es.strategy_type
        LIMIT 5
    """)]

sids = [s["sid"] for s in strategies]
cmp = compare_engineering_strategies(driver, sids)

for s in cmp["strategies"]:
    print(f"\n=== {s['strategy_type']} ({s['strategy_id']}) ===")
    print(f"  构建体: {s['construct_count']}")
    print(f"  目标能力: {s['intended_capabilities']}")
    print(f"  证据成熟度: {s['evidence_maturity']}")
    print(f"  组织: {s['known_organizations']}")
    print(f"  专利: {len(s['associated_patents'])}")
    print(f"  内部能力差距: {s['internal_capability_gaps']}")

print("\n=== comparison_matrix ===")
print(json.dumps(cmp["comparison_matrix"], ensure_ascii=False, indent=2))


=== biofilm_disruption (ENG:STRAT:B7FD45B5) ===
  构建体: 2
  目标能力: ['biofilm_disruption']
  证据成熟度: in_vitro
  组织: ['Merck KGaA']
  专利: 3
  内部能力差距: [{'level': 'high'}, {'level': 'high'}, {'level': 'medium'}]

=== host_range_expansion (ENG:STRAT:33F089EB) ===
  构建体: 2
  目标能力: ['host_range_expansion']
  证据成熟度: in_vitro
  组织: ['Merck KGaA']
  专利: 3
  内部能力差距: [{'level': 'medium'}, {'level': 'medium'}, {'level': 'medium'}]

=== lysis_enhancement (ENG:STRAT:5F826A3C) ===
  构建体: 2
  目标能力: ['lysis_enhancement']
  证据成熟度: in_vitro
  组织: ['Merck KGaA']
  专利: 3
  内部能力差距: [{'level': 'medium'}, {'level': 'medium'}, {'level': 'medium'}]

=== payload_delivery (ENG:STRAT:A4D5D7F3) ===
  构建体: 2
  目标能力: ['payload_delivery']
  证据成熟度: in_vitro
  组织: ['Merck KGaA']
  专利: 3
  内部能力差距: [{'level': 'high'}, {'level': 'high'}, {'level': 'medium'}]

=== tail_fiber_engineering (ENG:STRAT:CE258659) ===
  构建体: 2
  目标能力: ['tail_fiber_engineering']
  证据成熟度: in_vitro
  组织: ['Merck KGaA']
  专利: 3
  内部能力差距: [{'level': 'medi

In [6]:
# ============================================================
# Cell 6: PRD 13.7 White-space 分析（全量范围）
# ============================================================
white = identify_landscape_white_space(driver)

print("=== summary ===")
print(json.dumps(white["summary"], ensure_ascii=False, indent=2))

print("\n=== top 10 underrepresented ===")
for c in white["underrepresented_combinations"][:10]:
    print(f"  {c['strategy_type']:30s} × {c['pathogen_species']:30s} "
          f"constructs={c['construct_count']} evidence={c['evidence_density']} "
          f"patents={c['patent_density']} orgs={c['known_competitor_count']} [{c['coverage']}]")

print("\n=== disclaimer ===")
print(white["disclaimer"])

=== summary ===
{
  "total_combinations": 30,
  "underrepresented_count": 30,
  "empty_count": 20,
  "minimal_count": 10,
  "sparse_count": 0,
  "covered_count": 0
}

=== top 10 underrepresented ===
  host_range_expansion           × Staphylococcus aureus          constructs=0 evidence=0 patents=0 orgs=0 [empty]
  host_range_expansion           × Escherichia coli               constructs=0 evidence=0 patents=0 orgs=0 [empty]
  host_range_expansion           × Acinetobacter baumannii        constructs=0 evidence=0 patents=0 orgs=0 [empty]
  host_range_expansion           × Proteus mirabilis              constructs=0 evidence=0 patents=0 orgs=0 [empty]
  tail_fiber_engineering         × Staphylococcus aureus          constructs=0 evidence=0 patents=0 orgs=0 [empty]
  tail_fiber_engineering         × Pseudomonas aeruginosa         constructs=0 evidence=0 patents=0 orgs=0 [empty]
  tail_fiber_engineering         × Acinetobacter baumannii        constructs=0 evidence=0 patents=0 orgs=0 [emp

In [7]:
# ============================================================
# Cell 7: PRD 13.7 White-space 分析（带筛选条件）
# ============================================================
white2 = identify_landscape_white_space(
    driver,
    pathogen_scope=["Klebsiella pneumoniae", "Pseudomonas aeruginosa"],
    technology_scope=["host_range_expansion", "lysis_enhancement", "biofilm_disruption"],
)

print("=== 筛选后 summary ===")
print(json.dumps(white2["summary"], ensure_ascii=False, indent=2))

print("\n=== 筛选后 underrepresented 组合 ===")
for c in white2["underrepresented_combinations"]:
    print(f"  {c['strategy_type']:30s} × {c['pathogen_species']:30s} [{c['coverage']}]")

=== 筛选后 summary ===
{
  "total_combinations": 6,
  "underrepresented_count": 6,
  "empty_count": 2,
  "minimal_count": 4,
  "sparse_count": 0,
  "covered_count": 0
}

=== 筛选后 underrepresented 组合 ===
  lysis_enhancement              × Klebsiella pneumoniae          [empty]
  biofilm_disruption             × Klebsiella pneumoniae          [empty]
  host_range_expansion           × Klebsiella pneumoniae          [minimal]
  host_range_expansion           × Pseudomonas aeruginosa         [minimal]
  lysis_enhancement              × Pseudomonas aeruginosa         [minimal]
  biofilm_disruption             × Pseudomonas aeruginosa         [minimal]
